# Product Detective — ML Exploration Notebook

This notebook is for **exploratory analysis** before training.
It covers:
1. Review data exploration & visualisation
2. Sentiment distribution analysis
3. Complaint cluster quality check
4. Trust score feature analysis
5. Decision model feature importance
6. Error analysis on misclassified reviews

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from datetime import datetime

# Product Detective modules
from modules.sentiment_model import SentimentModel
from modules.complaint_detector import ComplaintDetector
from modules.complaint_trend_analyzer import ComplaintTrendAnalyzer
from modules.review_trust_model import ReviewTrustModel
from modules.category_classifier import SpecEvaluator
from modules.decision_engine import DecisionEngine

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'monospace'

print('✅ Imports OK')

## 1. Load Sample Data

In [ ]:
# Load labelled data (or create synthetic for demo)
try:
    df = pd.read_csv('../ml/data/complaint_labels.csv')
    print(f'Loaded {len(df)} labelled reviews')
except FileNotFoundError:
    print('No labelled data found — generating synthetic samples')
    from ml.training.train_pipeline import _generate_synthetic_complaint_data
    df = _generate_synthetic_complaint_data()

print(df.head())
print('\nCategory distribution:')
print(df['complaint_category'].value_counts())

## 2. Sentiment Distribution

In [ ]:
model = SentimentModel()
model._pipe = None  # Force rule-based for notebook speed

reviews = df[['text']].rename(columns={'text': 'body'})
reviews['review_id'] = [f'R{i:04d}' for i in range(len(reviews))]
reviews_list = reviews[['review_id', 'body']].to_dict('records')

report = model.analyze_reviews(reviews_list)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
sizes = [report.positive_pct, report.neutral_pct, report.negative_pct]
labels = ['Positive', 'Neutral', 'Negative']
colors = ['#3A8F52', '#C8973A', '#D9412E']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Sentiment Distribution')

# Top phrases
all_phrases = report.top_positive_phrases[:6] + report.top_negative_phrases[:6]
phrase_types = ['positive'] * 6 + ['negative'] * 6
phrase_colors = ['#3A8F52' if t == 'positive' else '#D9412E' for t in phrase_types]
axes[1].barh(all_phrases, [1]*len(all_phrases), color=phrase_colors)
axes[1].set_title('Top Phrases by Sentiment')
axes[1].set_xlabel('Frequency rank')

plt.tight_layout()
plt.savefig('../ml/data/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Positive: {report.positive_pct:.1f}%')
print(f'Neutral:  {report.neutral_pct:.1f}%')
print(f'Negative: {report.negative_pct:.1f}%')
print(f'Avg score: {report.avg_score:.3f}')

## 3. Complaint Cluster Analysis

In [ ]:
detector = ComplaintDetector()
reviews_full = [
    {'review_id': r['review_id'], 'body': r['body'], 'rating': 2}
    for r in reviews_list
]
sentiment_map = {r['review_id']: 'negative' for r in reviews_list}

complaint_report = detector.detect(reviews_full, 'laptop', sentiment_map)

if complaint_report.clusters:
    cluster_df = pd.DataFrame([
        {
            'Category':  c.label,
            'Total %':   c.total_pct,
            'Severity':  c.severity,
        }
        for c in complaint_report.clusters
    ]).sort_values('Total %', ascending=True)

    severity_colors = {'critical': '#D9412E', 'moderate': '#C8973A', 'minor': '#888'}
    colors = [severity_colors.get(s, '#888') for s in cluster_df['Severity']]

    fig, ax = plt.subplots(figsize=(10, max(4, len(cluster_df) * 0.6)))
    bars = ax.barh(cluster_df['Category'], cluster_df['Total %'], color=colors)
    ax.set_xlabel('% of all reviews')
    ax.set_title('Complaint Frequency by Category')
    ax.axvline(x=15, color='red', linestyle='--', alpha=0.5, label='Critical threshold (15%)')
    ax.legend()

    # Add value labels
    for bar, val in zip(bars, cluster_df['Total %']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('../ml/data/complaint_clusters.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No complaint clusters found')

## 4. Trust Score Feature Importance

In [ ]:
trust_model = ReviewTrustModel()

# Build test scenarios
scenarios = [
    ('Authentic (verified, diverse dates)', 85),
    ('Burst pattern (all same week)',       55),
    ('Duplicates (copy-paste reviews)',     60),
    ('Rating mismatch (5★ negative text)', 65),
    ('Unverified (all unverified)',         70),
]

labels_s = [s[0] for s in scenarios]
scores_s = [s[1] for s in scenarios]

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#3A8F52' if s >= 80 else '#C8973A' if s >= 60 else '#D9412E' for s in scores_s]
bars = ax.barh(labels_s, scores_s, color=bar_colors)
ax.set_xlim(0, 100)
ax.axvline(x=80, color='green', linestyle='--', alpha=0.6, label='High trust threshold (80)')
ax.axvline(x=60, color='orange', linestyle='--', alpha=0.6, label='Low trust threshold (60)')
ax.set_xlabel('Trust Score (0–100)')
ax.set_title('Trust Score by Review Pattern Scenario')
ax.legend()

for bar, val in zip(bars, scores_s):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../ml/data/trust_score_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Decision Engine Composite Score Analysis

In [ ]:
from ml.training.train_pipeline import _generate_synthetic_verdict_data

verdict_df = _generate_synthetic_verdict_data()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

verdicts = ['BUY', 'WAIT', 'AVOID']
colors = {'BUY': '#3A8F52', 'WAIT': '#C8973A', 'AVOID': '#D9412E'}

# Feature distributions by verdict
features_to_plot = [
    ('positive_pct', 'Positive Review %'),
    ('trust_score',  'Trust Score'),
    ('spec_score',   'Spec Score'),
]

for ax, (feat, label) in zip(axes, features_to_plot):
    for verdict in verdicts:
        subset = verdict_df[verdict_df['verdict'] == verdict][feat]
        ax.hist(subset, bins=15, alpha=0.6, label=verdict,
                color=colors[verdict], edgecolor='white')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'{label} by Verdict')
    ax.legend()

plt.suptitle('Feature Distributions by Verdict Class', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../ml/data/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print(verdict_df.groupby('verdict')[['positive_pct','trust_score','spec_score',
                                      'critical_complaint_count']].mean().round(1))

## 6. Complaint Timeline Simulation

In [ ]:
import matplotlib.dates as mdates
from datetime import timedelta

# Simulate a rising complaint trend
months = pd.date_range('2024-10-01', periods=6, freq='MS')
overheat_pcts = [8, 11, 16, 22, 29, 34]   # rising
battery_pcts  = [18, 19, 18, 20, 19, 21]  # stable

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(months, overheat_pcts, 'o-', color='#D9412E', linewidth=2.5,
        markersize=8, label='Overheating (RISING ↑)')
ax.plot(months, battery_pcts,  's--', color='#C8973A', linewidth=1.5,
        markersize=6, label='Battery (Stable)')

ax.axhline(y=15, color='red', linestyle=':', alpha=0.5, label='Critical threshold (15%)')
ax.fill_between(months, overheat_pcts, alpha=0.08, color='#D9412E')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.set_ylabel('% of reviews mentioning complaint')
ax.set_title('Complaint Trend Analysis — Last 6 Months')
ax.legend()
ax.set_ylim(0, 45)

# Annotate final values
ax.annotate(f'{overheat_pcts[-1]}%', xy=(months[-1], overheat_pcts[-1]),
            xytext=(10, 5), textcoords='offset points',
            color='#D9412E', fontweight='bold')

plt.tight_layout()
plt.savefig('../ml/data/complaint_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# Linear regression stats
from scipy import stats
x = np.arange(len(overheat_pcts))
slope, intercept, r, p, se = stats.linregress(x, overheat_pcts)
print(f'Overheating trend: slope={slope:.2f} pp/month, R²={r**2:.3f}, p={p:.4f}')
print(f'→ Significant: {"YES ⚠" if p < 0.05 and slope > 2 else "NO"}')

## 7. Model Training & Evaluation

In [ ]:
import sys
sys.path.insert(0, '..')
from ml.training.train_pipeline import train_complaint_classifier, train_verdict_predictor

print('Training complaint classifier...')
c_results = train_complaint_classifier()
print(f'  CV F1: {c_results["cv_f1"]:.4f}')
print(f'  Test F1: {c_results["test_f1"]:.4f}')

print('\nTraining verdict predictor...')
v_results = train_verdict_predictor()
print(f'  CV F1: {v_results["cv_f1"]:.4f}')
print(f'  Test F1: {v_results["test_f1"]:.4f}')
print(f'  Algorithm: {v_results["algorithm"]}')

## 8. End-to-End Pipeline Smoke Test

In [ ]:
from unittest.mock import MagicMock

# Build mock reports to test the decision engine
def mock_sentiment(pos, neu, neg):
    r = MagicMock()
    r.positive_pct, r.neutral_pct, r.negative_pct = pos, neu, neg
    r.avg_score = (pos - neg) / 100
    return r

def mock_complaints(n_critical=0, n_moderate=0):
    r = MagicMock()
    r.clusters = [
        MagicMock(severity='critical', label='Overheating', total_pct=25.0)
        for _ in range(n_critical)
    ] + [
        MagicMock(severity='moderate', label='Battery', total_pct=12.0)
        for _ in range(n_moderate)
    ]
    return r

def mock_trend(critical=False):
    r = MagicMock(); r.has_critical_trend = critical; r.signals = []; r.summary = ''
    return r

def mock_trust(score): r = MagicMock(); r.score = score; r.summary = ''; return r
def mock_specs(s):     r = MagicMock(); r.overall_spec_score = s; r.strengths=[]; r.weaknesses=[]; r.spec_scores=[]; return r

engine = DecisionEngine()

test_cases = [
    ('BUY scenario',   mock_sentiment(75,15,10), mock_complaints(0,0), mock_trend(False), mock_trust(88), mock_specs(80)),
    ('WAIT scenario',  mock_sentiment(52,20,28), mock_complaints(1,1), mock_trend(True),  mock_trust(68), mock_specs(58)),
    ('AVOID scenario', mock_sentiment(30,15,55), mock_complaints(3,1), mock_trend(True),  mock_trust(45), mock_specs(35)),
]

print(f"{'Scenario':<20} {'Verdict':<8} {'Confidence':<12} {'Score':<8}")
print('-' * 50)
for name, s, c, t, tr, sp in test_cases:
    result = engine.decide(s, c, t, tr, sp)
    print(f"{name:<20} {result.verdict.value:<8} {result.confidence:<12} {result.composite_score:.3f}")

print('\n✅ Pipeline smoke test passed')